# Chapter 3: Classification

In this chapter, we will build a classification system using the famous **MNIST dataset**. It contains 70,000 small images of handwritten digits (0-9). 

## 1. Fetching the MNIST Dataset
Scikit-Learn provides helper functions to easily download popular datasets.

```python
from sklearn.datasets import fetch_openml
import numpy as np

# Fetch the dataset from OpenML
# as_frame=False ensures we get numpy arrays instead of Pandas DataFrames (easier for image data)
mnist = fetch_openml('mnist_784', version=1, as_frame=False)

# Separate the features (pixel data) and the target (labels)
X, y = mnist["data"], mnist["target"]

# Display the shape of our dataset
print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

## 2. Visualizing the Data
Since these are images, it's always a good idea to look at them. We can use `matplotlib` to grab a single row (which has 784 pixels), reshape it back into a 28x28 grid, and display it.

```python
import matplotlib.pyplot as plt

def plot_digit(image_data):
    # Reshape the 784 array back to a 28x28 image grid
    image = image_data.reshape(28, 28)
    plt.imshow(image, cmap="binary")
    plt.axis("off")

# Let's look at the very first digit in our dataset
some_digit = X[0]
plot_digit(some_digit)
plt.show()

# If we check the label for this image, it should match!
print(f"The label for this digit is: {y[0]}")
```

## 3. Creating the Test Set
Just like in Chapter 2, we must set aside a test set before we start inspecting the data too closely or training our models. The MNIST dataset is already split into a training set (the first 60,000 images) and a test set (the last 10,000 images).

```python
# Split the data sequentially since the dataset is already pre-shuffled for us
X_train, X_test, y_train, y_test = X[:60000], X[60000:], y[:60000], y[60000:]
```

## 4. Training a Binary Classifier
Before trying to classify all 10 digits, let's simplify the problem and train a model to only identify one digit: the number 5. This will be a "5-detector", which is an example of a Binary Classifier (it only answers True or False).

First, we need to create the target vectors for this classification task:
```python
# Create a new target array that is True when the label is '5', and False for all other digits.
y_train_5 = (y_train == '5')
y_test_5 = (y_test == '5')
```
Now, let's pick a classifier and train it. A good place to start is with a Stochastic Gradient Descent (SGD) classifier. It's fast and handles large datasets efficiently.

```python
from sklearn.linear_model import SGDClassifier

# Create the model (random_state ensures we get the same reproducible results)
sgd_clf = SGDClassifier(random_state=42)

# Train the model using our training features and our new True/False labels
sgd_clf.fit(X_train, y_train_5)
```

### What is SGDClassifier?
`SGDClassifier` stands for **Stochastic Gradient Descent Classifier**. 
It is not a completely new algorithm, but rather a simple linear model that uses the **Stochastic Gradient Descent** optimization technique. Instead of calculating the error on the entire dataset at once (which is very slow), it processes training instances one by one, randomly. This makes it incredibly fast and well-suited for huge datasets like MNIST.

Let's test our newly trained model on that first digit we visualized earlier:

```python
# Pass the single digit (inside a list because predict expects a 2D array)
# It should output [True] because we saw earlier that it is a 5!
predictions = sgd_clf.predict([some_digit])
print(f"Prediction for the first digit: {predictions}")
```

## 5. Performance Measures
Evaluating a classifier is often significantly trickier than evaluating a regressor. 

### Measuring Accuracy Using Cross-Validation
We can use `cross_val_score` to evaluate our `SGDClassifier` model, just like we did in Chapter 2.

```python
from sklearn.model_selection import cross_val_score

# Test our model using 3-fold cross-validation
cross_val_score(sgd_clf, X_train, y_train_5, cv=3, scoring="accuracy")
```
*Output is around 95% or 96% accuracy! Sounds amazing, right? Let's check if it actually is.*

### The Flaw of Accuracy (The Dummy Classifier)
Let's look at a very "dumb" classifier that just classifies every single image in the most frequent class, which in our case is "not-5" (False).

```python
from sklearn.dummy import DummyClassifier

# Create a dummy classifier that always predicts the most frequent class
dummy_clf = DummyClassifier()
dummy_clf.fit(X_train, y_train_5)

# Let's see its accuracy
cross_val_score(dummy_clf, X_train, y_train_5, cv=3, scoring="accuracy")
```
*Output is over 90%! This is because only about 10% of the images are 5s. So if you always guess that an image is not a 5, you will be right about 90% of the time. This demonstrates why accuracy is generally not the preferred performance measure for classifiers, especially when you are dealing with skewed datasets.*

## 6. Confusion Matrix
A much better way to evaluate the performance of a classifier is to look at the confusion matrix. The general idea is to count the number of times instances of class A are classified as class B.

First, we need a set of predictions so they can be compared to the actual targets:
```python
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

# Get clean predictions for every instance in the training set
y_train_pred = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3)

# Generate the confusion matrix
cm = confusion_matrix(y_train_5, y_train_pred)
print(cm)
```

### Precision, Recall, and F1 Score
Scikit-Learn provides functions to compute classifier metrics based on the confusion matrix:
*   **Precision:** Accuracy of the positive predictions (When it claims an image is a 5, how often is it correct?).
*   **Recall:** True Positive Rate (Out of all the actual 5s in the dataset, how many did the classifier detect?).
*   **F1 Score:** The harmonic mean of precision and recall.

```python
from sklearn.metrics import precision_score, recall_score, f1_score

print(f"Precision: {precision_score(y_train_5, y_train_pred):.4f}")
print(f"Recall: {recall_score(y_train_5, y_train_pred):.4f}")
print(f"F1 Score: {f1_score(y_train_5, y_train_pred):.4f}")
```

## 7. Precision/Recall Trade-off
How does `SGDClassifier` make its decisions? For each instance, it computes a score based on a *decision function*. If that score is greater than a certain threshold, it assigns the instance to the positive class; otherwise, it assigns it to the negative class.

Scikit-Learn does not let you set the threshold directly, but it does give you access to the decision scores!

```python
# 1. Get the decision scores instead of the True/False predictions
y_scores = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3, method="decision_function")

# 2. Compute precision and recall for all possible thresholds
from sklearn.metrics import precision_recall_curve
precisions, recalls, thresholds = precision_recall_curve(y_train_5, y_scores)
```

Suppose your project requires exactly 90% precision. You can search for the lowest threshold that gives you at least 90% precision using `numpy`:

```python
import numpy as np
from sklearn.metrics import precision_score, recall_score

# 3. Find the exact threshold for 90% precision
idx_for_90_precision = (precisions >= 0.90).argmax()
threshold_for_90_precision = thresholds[idx_for_90_precision]

# 4. Make new predictions based on this custom threshold
y_train_pred_90 = (y_scores >= threshold_for_90_precision)

# 5. Check our new precision and recall
print(f"New Precision: {precision_score(y_train_5, y_train_pred_90):.4f}")
print(f"New Recall: {recall_score(y_train_5, y_train_pred_90):.4f}")
```
*As you can see, you can create a classifier with virtually any precision you want by just setting a high enough threshold. But remember: a high-precision classifier is not very useful if its recall is too low!*

## 8. The ROC Curve and AUC
The Receiver Operating Characteristic (ROC) curve plots the True Positive Rate (Recall) against the False Positive Rate (FPR). 

```python
from sklearn.metrics import roc_curve, roc_auc_score

# Get FPR, TPR, and thresholds
fpr, tpr, thresholds = roc_curve(y_train_5, y_scores)

# Calculate the Area Under the Curve (AUC)
roc_auc = roc_auc_score(y_train_5, y_scores)
print(f"SGD ROC AUC Score: {roc_auc:.4f}")
```

## 9. Training a Random Forest Classifier
Unlike `SGDClassifier`, the `RandomForestClassifier` does not have a `decision_function()` method. Instead, it has a `predict_proba()` method that returns a list of probabilities for each instance (e.g., 89% chance that it is a 5).

```python
from sklearn.ensemble import RandomForestClassifier

# 1. Initialize the Random Forest model
forest_clf = RandomForestClassifier(random_state=42)

# 2. Get probabilities using cross-validation
y_probas_forest = cross_val_predict(forest_clf, X_train, y_train_5, cv=3, method="predict_proba")

# 3. Grab the probabilities for the positive class (being a 5)
y_scores_forest = y_probas_forest[:, 1]
```

### Evaluating Random Forest Metrics
We can now use these probabilities to plot the Precision/Recall curve or compute the F1 score using the standard 50% threshold:

```python
from sklearn.metrics import precision_score, recall_score, f1_score

y_train_pred_forest = y_scores_forest >= 0.5

print(f"Random Forest F1 Score: {f1_score(y_train_5, y_train_pred_forest):.4f}")
print(f"Random Forest Precision: {precision_score(y_train_5, y_train_pred_forest):.4f}")
print(f"Random Forest Recall: {recall_score(y_train_5, y_train_pred_forest):.4f}")
```

## 10. Multiclass Classification
Binary classifiers distinguish between two classes, but multiclass classifiers can distinguish between more than two. Some algorithms (like Random Forest) handle this natively. Others (like SVM or SGD) are strictly binary, but Scikit-Learn provides clever strategies to use them for multiclass tasks:

*   **One-versus-the-Rest (OvR):** Train 10 binary classifiers (one for each digit). When you want to classify an image, you get the decision score from all 10 and pick the highest.
*   **One-versus-One (OvO):** Train a binary classifier for every pair of digits (e.g., 0s vs 1s, 0s vs 2s). For 10 classes, this means training 45 models! 

Scikit-Learn detects when you try to use a binary classification algorithm for a multiclass task and automatically runs OvR or OvO. Let's try it with a Support Vector Classifier (SVC). 
*Note: We only use the first 2,000 instances because SVMs scale poorly to large datasets.*

```python
from sklearn.svm import SVC

# Scikit-Learn automatically uses OvO under the hood for SVM
svm_clf = SVC(random_state=42)
# Notice we use y_train (all digits 0-9), not y_train_5!
svm_clf.fit(X_train[:2000], y_train[:2000]) 

# Let's test it on our favorite digit
print(f"Prediction: {svm_clf.predict([some_digit])}")

# We can see the 10 scores (one for each class)
some_digit_scores = svm_clf.decision_function([some_digit])
print(f"Scores: \n{some_digit_scores.round(2)}")
```

If you specifically want to force Scikit-Learn to use OvR, you can explicitly use the `OneVsRestClassifier`:
```python
from sklearn.multiclass import OneVsRestClassifier

ovr_clf = OneVsRestClassifier(SVC(random_state=42))
ovr_clf.fit(X_train[:2000], y_train[:2000])
print(f"Number of trained models: {len(ovr_clf.estimators_)}") # Should output 10
```

### Training an SGD Multiclass Model & Scaling
Now let's train our `SGDClassifier` on the entire dataset. 
```python
# Train SGD on all 10 classes
sgd_clf = SGDClassifier(random_state=42)
sgd_clf.fit(X_train, y_train)

# Let's check its overall accuracy using Cross-Validation
from sklearn.model_selection import cross_val_score
cross_val_score(sgd_clf, X_train, y_train, cv=3, scoring="accuracy")
```
*The accuracy is around 87%. But we can do better! Just like in Chapter 2, scaling the inputs helps algorithms perform better and converge faster.*

```python
from sklearn.preprocessing import StandardScaler

# Scale the pixel values
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.astype("float64"))

# Check accuracy again after scaling
cross_val_score(sgd_clf, X_train_scaled, y_train, cv=3, scoring="accuracy")
```
*Accuracy jumps to nearly 90% just by scaling the data!*

### Key Concepts

**1. What is Support Vector Machine (SVM)?**
SVM is a very powerful machine learning model. If you imagine wanting to separate two classes of data with a line, a standard linear model just draws any line that separates them. An SVM, however, tries to draw the "widest possible street" between the classes. It looks for the largest margin of safety. While highly accurate, SVMs can be very slow on large datasets.

**2. Binary vs. Multiclass Models**
*   **Inherently Multiclass:** Models like Random Forest can directly classify instances into multiple categories (e.g., distinguishing between 0-9 all at once).
*   **Inherently Binary:** Models like SGD and SVM can only distinguish between two classes (e.g., "is a 5" or "not a 5"). 

**3. Multiclass Strategies for Binary Classifiers**
To use binary models for multiclass tasks (like our 10 digits), Scikit-Learn uses clever workarounds:
*   **OvR (One-versus-the-Rest):** Trains 10 binary classifiers (is it 0 or not? is it 1 or not?). It gets a score from each and picks the highest.
*   **OvO (One-versus-One):** Trains a binary classifier for every single pair of classes (0 vs 1, 0 vs 2, etc.). For 10 classes, it requires 45 models! The class that wins the most duels is the final prediction. Scikit-Learn automatically uses OvO for SVMs to save time (since SVMs train faster on many small datasets rather than one huge dataset).

**4. Why Scale Data?**
Algorithms like SGD rely on calculating distances and gradients. If pixel values range from 0 to 255, the model can get confused by the large numbers. `StandardScaler` squashes these numbers down (so they have a mean of 0 and variance of 1) without changing the underlying pattern of the image. This makes the math much easier for the algorithm, often resulting in a significant accuracy boost!

## 11. Error Analysis
To figure out how to improve your model, it's very helpful to analyze the types of errors it makes. We can do this visually using the Confusion Matrix.

```python
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. Generate cross-validation predictions
y_train_pred = cross_val_predict(sgd_clf, X_train_scaled, y_train, cv=3)

# 2. Plot the normalized confusion matrix (shows percentages)
ConfusionMatrixDisplay.from_predictions(y_train, y_train_pred, 
                                        normalize="true", values_format=".0%")
plt.title("Normalized Confusion Matrix")
plt.show()
```

### Focusing on the Errors
To see clearly where the model is messing up, we can zero out the correct predictions (the diagonal) and focus purely on the mistakes:

```python
# Create a weight array that is True (1) for errors and False (0) for correct predictions
sample_weight = (y_train_pred != y_train)

# Plot the matrix again using these weights
ConfusionMatrixDisplay.from_predictions(y_train, y_train_pred,
                                        sample_weight=sample_weight,
                                        normalize="true", values_format=".0%")
plt.title("Errors Only")
plt.show()
```

## 12. Multilabel Classification
Until now, each instance has always been assigned to just one class. In some cases, you may want your classifier to output multiple classes for each instance (e.g., face recognition identifying multiple people in one photo). This is called multilabel classification.

Let's create a `y_multilabel` array containing two target labels for each digit image:
1. Is the digit large (7, 8, or 9)?
2. Is the digit odd?

```python
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

# Create the two conditions
y_train_large = (y_train >= '7')
y_train_odd = (y_train.astype('int8') % 2 == 1)

# Combine them into a single array with two columns
y_multilabel = np.c_[y_train_large, y_train_odd]

# Train a KNN classifier (which natively supports multilabel classification)
knn_clf = KNeighborsClassifier()
knn_clf.fit(X_train, y_multilabel)
```

Now let's make a prediction on our favorite digit (the number 5):
```python
knn_clf.predict([some_digit])
```
*Output is `array([[False, True]])`. This is correct! The number 5 is not large (False), but it is odd (True).*

### Evaluating a Multilabel Classifier
To evaluate a multilabel classifier, you can measure the F1 score for each individual label (e.g., one score for "large" and one for "odd"), and then simply compute the average score.

```python
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import f1_score

# Warning: This cell may take a few minutes to run!
y_train_knn_pred = cross_val_predict(knn_clf, X_train, y_multilabel, cv=3)

# Calculate the macro average F1 score
macro_f1 = f1_score(y_multilabel, y_train_knn_pred, average="macro")
print(f"Macro F1 Score: {macro_f1:.4f}")

# Calculate the weighted average F1 score
weighted_f1 = f1_score(y_multilabel, y_train_knn_pred, average="weighted")
print(f"Weighted F1 Score: {weighted_f1:.4f}")
```

## 13. Multioutput Classification & The KNN Algorithm

Before diving into the code, let's fully understand the **K-Nearest Neighbors (KNN)** algorithm and how it relates to our final task.

### How KNN Works
Unlike other algorithms (like SGD) that try to draw a separating line or find mathematical weights, KNN relies purely on memory and geometry:
1. **Data as Points:** It treats every image in our dataset as a single point in a massive 784-dimensional space (since each image has 784 pixels).
2. **Measuring Distance:** When you give it a new, unseen image, it calculates the direct physical distance between this new image and all 60,000 images in its memory using the **Euclidean distance** (which is essentially the Pythagorean theorem extended to multiple dimensions).
3. **Voting:** It finds the "K" closest images (e.g., the 3 or 5 nearest neighbors). These neighbors then "vote" to determine the final answer for the new image.

### What is Multioutput Classification?
In this final section, we are building a **Noise Removal** system. This task is called Multioutput Classification because it combines two concepts:
* We are asking **784 separate questions** (one for every single pixel in the image).
* Each question has **256 possible answers** (the pixel intensity can be anything from 0 to 255).
It is a combination of Multilabel (multiple questions) and Multiclass (multiple options per question).

### The Code Implementation

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier

# 1. Generate random noise and add it to both training and test sets
rng = np.random.default_rng(seed=42) # Ensures reproducibility
noise_train = rng.integers(0, 100, (len(X_train), 784))
X_train_mod = X_train + noise_train

noise_test = rng.integers(0, 100, (len(X_test), 784))
X_test_mod = X_test + noise_test

# 2. Set the target labels to be the original, clean images!
# We want the model to learn how to map a noisy image back to its clean version.
y_train_mod = X_train
y_test_mod = X_test

# 3. Train the KNN classifier
# The model stores the noisy images and their clean counterparts.
knn_clf = KNeighborsClassifier()
knn_clf.fit(X_train_mod, y_train_mod)

# 4. Predict (clean up) a single noisy test image
clean_digit = knn_clf.predict([X_test_mod[0]])

# Note: You can visualize the cleaned digit using Matplotlib:
# plt.imshow(clean_digit.reshape(28, 28), cmap="binary")
# plt.show()
```

### Understanding the Output
Because our target (`y_train_mod`) was an array of 784 pixels, the final prediction output (`clean_digit`) will be an array containing exactly **784 numbers**. We simply reshape these 784 numbers back into a 28x28 grid to see the beautifully restored image!

## Exercise 1: An MNIST Classifier With Over 97% Accuracy

The goal of this exercise is to build a classifier for the MNIST dataset that achieves over 97% accuracy on the test set. We will use the `KNeighborsClassifier` and find the optimal hyperparameters using Grid Search.

### 1. The Baseline Model
First, let's establish a baseline by evaluating how a standard KNN model performs with its default settings.

```python
from sklearn.neighbors import KNeighborsClassifier

# Initialize and train the standard KNN model
knn_clf = KNeighborsClassifier()
knn_clf.fit(X_train, y_train)

# Measure baseline accuracy on the test set
baseline_accuracy = knn_clf.score(X_test, y_test)
print(f"Baseline Accuracy: {baseline_accuracy}") 
# Expected Output: ~0.9688
```
*The default KNN is already very close to our 97% goal!*

### 2. Hyperparameter Tuning with Grid Search
To squeeze out more accuracy, we can test different combinations of hyperparameters (like the number of neighbors and the weight function). To speed up this intensive search, we will only train on the first 10,000 images.

```python
from sklearn.model_selection import GridSearchCV

# Define the parameter grid to explore
param_grid = [{'weights': ["uniform", "distance"], 'n_neighbors': [3, 4, 5, 6]}]

# Set up the Grid Search with 5-fold cross-validation
grid_search = GridSearchCV(knn_clf, param_grid, cv=5)
        
# Fit the search on a smaller subset (10,000 instances) to save time
grid_search.fit(X_train[:10_000], y_train[:10_000])

print(f"Best Parameters: {grid_search.best_params_}")
# Expected Output: {'n_neighbors': 4, 'weights': 'distance'}

print(f"Best Score on Subset: {grid_search.best_score_}")
# Expected Output: ~0.944
```

### 3. Training the Best Model on the Full Dataset
Now that we have found the best hyperparameters, let's train this optimized model on the complete training set and evaluate it on the test set.

```python
# Train the best estimator on the full training set
grid_search.best_estimator_.fit(X_train, y_train)

# Evaluate the tuned model on the test set
tuned_accuracy = grid_search.best_estimator_.score(X_test, y_test)
print(f"Tuned Accuracy: {tuned_accuracy}")
# Expected Output: ~0.9714
```
*We successfully reached our goal of >97% accuracy!*

## Exercise 2: Data Augmentation (Training Set Expansion)

**What is Data Augmentation?**
Machine learning models need a massive amount of data to learn effectively. If we don't have new images to show our model, we can artificially generate them by making small, realistic modifications to our existing training data. 

For example, if we take an image of the number '5' and shift it one pixel to the left, it is still undeniably a '5'. By training the model on the original images *plus* all these shifted variations, we force the algorithm to learn the actual shape of the digits, regardless of where they are positioned in the frame. This technique is called **Data Augmentation** and it makes our model significantly more robust and accurate.

### The Code Implementation

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import shift
from sklearn.neighbors import KNeighborsClassifier

# 1. Create a function to shift an image by given pixels (dx, dy)
def shift_image(image, dx, dy):
    # Reshape the 1D array (784,) back to a 2D image grid (28, 28)
    image = image.reshape((28, 28))
    # The shift function moves the pixels and fills new empty spaces with 0 (black)
    shifted_image = shift(image, [dy, dx], cval=0, mode="constant")
    # Flatten it back to a 1D array for the machine learning model
    return shifted_image.reshape([-1])

# 2. Create the augmented dataset
X_train_augmented = [image for image in X_train]
y_train_augmented = [label for label in y_train]

# For every single image, create 4 shifted copies (left, right, down, up)
for dx, dy in ((-1, 0), (1, 0), (0, 1), (0, -1)):
    for image, label in zip(X_train, y_train):
        X_train_augmented.append(shift_image(image, dx, dy))
        y_train_augmented.append(label)

# Convert the python lists back into NumPy arrays for performance
X_train_augmented = np.array(X_train_augmented)
y_train_augmented = np.array(y_train_augmented)

# 3. Shuffle the newly expanded dataset
rng = np.random.default_rng(seed=42)
shuffle_idx = rng.permutation(len(X_train_augmented))
X_train_augmented = X_train_augmented[shuffle_idx]
y_train_augmented = y_train_augmented[shuffle_idx]

# 4. Train the tuned KNN model on the new massive dataset
# Note: **grid_search.best_params_ unpacks the best settings from Exercise 1
knn_clf = KNeighborsClassifier(**grid_search.best_params_)
knn_clf.fit(X_train_augmented, y_train_augmented)

# 5. Evaluate the final model
augmented_accuracy = knn_clf.score(X_test, y_test)
print(f"Augmented Accuracy: {augmented_accuracy}")
# Expected Output: ~0.9763 (The error rate dropped significantly!)
```

## Exercise 3: Tackle the Titanic Dataset

**The Goal:** Predict whether a passenger survived (1) or did not survive (0) based on attributes such as age, sex, passenger class, and where they embarked.

Before we can train a model, we need to clean and preprocess the raw data. Machine learning algorithms expect numerical inputs and cannot handle missing values or raw text automatically. We will build **Pipelines** to automate this data preparation process.

### 1. Inspecting the Data

```python
# Assuming load_titanic_data() has fetched the train_data and test_data DataFrames
# Set 'PassengerId' as the index since it is just an identifier, not a useful feature for learning
train_data = train_data.set_index("PassengerId")
test_data = test_data.set_index("PassengerId")

# train_data.info() reveals that 'Age', 'Cabin', and 'Embarked' have missing (NaN) values.
# We will ignore 'Cabin' (too many missing values), 'Name', and 'Ticket' for this basic model.
```

### 2. Building the Preprocessing Pipelines
We need two different strategies: one for numerical data (like age and fare) and one for categorical text data (like sex and embarked location).

```python
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

# --- Pipeline for Numerical Attributes ---
# 1. SimpleImputer: Fills missing numerical values with the median of that specific column
# 2. StandardScaler: Scales the data so that features have a mean of 0 and a variance of 1
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# --- Pipeline for Categorical Attributes ---
# 1. OrdinalEncoder: Can be used to encode ordered categories
# 2. SimpleImputer: Fills missing categories with the most frequent value in that column
# 3. OneHotEncoder: Converts categories into binary (0 or 1) dummy columns
cat_pipeline = Pipeline([
    ("ordinal_encoder", OrdinalEncoder()),
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("cat_encoder", OneHotEncoder(sparse_output=False)),
])

# --- Combining the Pipelines ---
# We use ColumnTransformer to apply the appropriate pipeline to the correct columns
num_attribs = ["Age", "SibSp", "Parch", "Fare"]
cat_attribs = ["Pclass", "Sex", "Embarked"]

preprocess_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs),
])
```

### 3. Training and Comparing Models

Now that we have a powerful preprocessing pipeline, we can easily prepare our training data and feed it into different Machine Learning algorithms to see which one performs best.

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt

# 1. Transform the raw data into numerical features using our pipeline
X_train = preprocess_pipeline.fit_transform(train_data)
y_train = train_data["Survived"]

# 2. Train a Random Forest Classifier
forest_clf = RandomForestClassifier(n_estimators=100, random_state=42)
forest_scores = cross_val_score(forest_clf, X_train, y_train, cv=10)
print(f"Random Forest Mean Accuracy: {forest_scores.mean():.4f}")
# Expected Output: ~0.813

# 3. Train a Support Vector Classifier (SVC)
from sklearn.svm import SVC
svm_clf = SVC(gamma="auto")
svm_scores = cross_val_score(svm_clf, X_train, y_train, cv=10)
print(f"SVC Mean Accuracy: {svm_scores.mean():.4f}")
# Expected Output: ~0.824
```

To better visualize the difference between the two models, we can plot their accuracy scores across the 10 cross-validation folds using a boxplot:

```python
# 4. Compare models using a boxplot
plt.figure(figsize=(8, 4))
plt.plot([1]*10, svm_scores, ".")
plt.plot([2]*10, forest_scores, ".")
plt.boxplot([svm_scores, forest_scores], labels=("SVM", "Random Forest"))
plt.ylabel("Accuracy")
plt.show()
```
*Note: The SVC model performs slightly better overall. It has a higher median accuracy and less variance (a tighter box) compared to the Random Forest.*

### 4. Feature Engineering
To boost our model's accuracy even further, we can engineer new features from the existing ones. This helps the model discover patterns more easily.

```python
# Example A: Creating Age Buckets
# Instead of exact ages, grouping passengers into age brackets can highlight survival trends.
train_data["AgeBucket"] = train_data["Age"] // 15 * 15
print(train_data[["AgeBucket", "Survived"]].groupby(['AgeBucket']).mean())

# Example B: Relatives Onboard
# Combining Siblings/Spouses (SibSp) and Parents/Children (Parch) into a single feature.
train_data["RelativesOnboard"] = train_data["SibSp"] + train_data["Parch"]
print(train_data[["RelativesOnboard", "Survived"]].groupby(['RelativesOnboard']).mean())
```

# 🎯 Chapter 3: Classification - The Big Picture & Workflow

This section is a high-level summary of everything we covered in Chapter 3. In Machine Learning, you don't need to memorize exact code syntax, but you **must** understand the core concepts, evaluation metrics, and workflow.

## 1. Training a Binary Classifier
* **Concept:** The simplest form of classification where the goal is to distinguish between just two classes (e.g., identifying the digit "5" vs. "not-5").
* **Action:** We used the Stochastic Gradient Descent (`SGDClassifier`) and evaluated performance using custom Cross-Validation (`cross_val_score`).

## 2. Performance Measures (Beyond Accuracy)
* **The Accuracy Trap:** Accuracy is a terrible metric for skewed datasets (e.g., if 90% of data is non-5, a dummy model guessing "not-5" is 90% accurate but useless).
* **Confusion Matrix:** Counts true positives, false positives, true negatives, and false negatives to give a complete picture.
* **Precision & Recall:** 
  * *Precision:* Accuracy of positive predictions (minimizes false positives).
  * *Recall:* Sensitivity or true positive rate (minimizes false negatives).
* **Precision/Recall Trade-off & ROC Curves:** We adjusted decision thresholds and used ROC curves (AUC) to evaluate models across different threshold levels.

## 3. Multiclass Classification
* **Concept:** Handling tasks with more than two classes (e.g., classifying handwritten digits from 0 to 9).
* **Action:** Some algorithms (like Random Forests or KNN) handle multiclass natively, while others (like binary classifiers) use strategies like OvR (One-versus-Rest) or OvO (One-versus-One).

## 4. Error Analysis
* **Action:** Instead of just looking at scores, we visually inspected the normalized Confusion Matrix (`plt.matshow()`) to analyze where the model gets confused (e.g., mistaking 3 for 5) and focused data collection or feature engineering accordingly.

## 5. Multilabel & Multioutput Classification
* **Multilabel Classification:** Outputting multiple binary labels per instance (e.g., is the digit large? is it odd?). Evaluated using average F1 scores (macro vs. weighted).
* **Multioutput Classification:** Generalization of multilabel where each label can have multiple classes (e.g., an image denoising system that predicts pixel color values from 0 to 255 for all 784 pixels).

## 6. The K-Nearest Neighbors (KNN) Algorithm
* **Concept:** A non-parametric "lazy learner" that does not build mathematical equations during training; instead, it memorizes the entire dataset.
* **Mechanism:** Computes geometric distance (Euclidean distance based on the Pythagorean theorem) between a new point and all stored data, finds the $K$ nearest neighbors, and performs majority voting.

## 7. Chapter Exercises & Practical Applications
* **MNIST >97% Accuracy:** Used `GridSearchCV` to tune hyperparameters (`n_neighbors`, `weights`) for a `KNeighborsClassifier` on the MNIST dataset.
* **Data Augmentation:** Artificially expanded the training set by shifting existing images in all directions (left, right, up, down) to make the model more robust and reduce error rates.
* **Tackling the Titanic Dataset:** 
  * Built complete preprocessing pipelines using `Pipeline`, `SimpleImputer`, `StandardScaler`, and `OneHotEncoder` via `ColumnTransformer`.
  * Trained and compared powerful models like `RandomForestClassifier` and `SVC`.
  * Applied **Feature Engineering** (e.g., creating `AgeBucket` and `RelativesOnboard`) to boost model insights.